# StyleSync — Occasion Classifier

Predicts **Casual / Sports / Formal** from a clothing image. Same ResNet-50 + `layer4` fine-tuning recipe as `baseline_improved.ipynb`, but with:

- **3 classes** (`usage` field, dropped `Party`=24 rows and `Travel`=1 row — not learnable).
- **Trained on the full 23k dataset** (the balanced 1394-row subset doesn't carry enough occasion variety since it was balanced by subCategory).
- **Class-weighted CrossEntropyLoss** for the heavy imbalance (Casual 77% / Sports 15% / Formal 8%).
- Reads from `data/images/raw_images/` (uncropped — same decision as the category model).
- Saves to `models/resnet50_stylesync_occasion.pt`.

In [1]:
import sys, os, copy
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
USAGE_TO_LABEL = {
    "Casual": 0,
    "Sports": 1,
    "Formal": 2,
}
LABEL_TO_USAGE = {v: k for k, v in USAGE_TO_LABEL.items()}
CLASS_NAMES    = [LABEL_TO_USAGE[i] for i in range(len(USAGE_TO_LABEL))]
NUM_CLASSES    = len(USAGE_TO_LABEL)

### Load the dataset

`combined_df.csv` is filtered to the 3 valid usages and to rows whose image is on disk. Expecting ~23k rows.

In [3]:
IMG_DIR = Path("../data/images/raw_images")

df = pd.read_csv("../data/combined_df.csv")
df = df[df["usage"].isin(USAGE_TO_LABEL)].reset_index(drop=True)
df = df[df["id"].apply(lambda x: (IMG_DIR / f"{x}.jpg").exists())].reset_index(drop=True)

print(f"Total samples: {len(df):,}\n")
print(df["usage"].value_counts())

Total samples: 23,363

usage
Casual    18031
Sports     3558
Formal     1774
Name: count, dtype: int64


In [4]:
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class WardrobeDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = self.image_dir / f"{row['id']}.jpg"
        label = USAGE_TO_LABEL[row["usage"]]
        img   = Image.open(path).convert("RGB")
        return self.transform(img), label

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["usage"]
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,}\n")
print("Train usage distribution:")
print(train_df["usage"].value_counts())

BATCH_SIZE = 32
train_dataset = WardrobeDataset(train_df, IMG_DIR, transform=train_transform)
val_dataset   = WardrobeDataset(val_df,   IMG_DIR, transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

### Class weights for the imbalance

Casual is 77%, Sports 15%, Formal 8%. Without weighting, the model would mostly learn to predict Casual. Inverse-frequency weights make a wrong prediction on Formal/Sports cost proportionally more during training.

In [ ]:
freq = train_df["usage"].value_counts().reindex(CLASS_NAMES).fillna(0).values
weights_per_class = len(train_df) / (NUM_CLASSES * freq)

for name, f, w in zip(CLASS_NAMES, freq, weights_per_class):
    print(f"{name:<8} count={int(f):>6}  weight={w:.3f}")

class_weights = torch.tensor(weights_per_class, dtype=torch.float32)

### Model — same recipe as the category classifier

ResNet-50, `layer4` unfrozen + new 3-way FC head. Differential learning rates: `1e-4` on the unfrozen backbone, `1e-3` on the head. AdamW with `weight_decay=1e-4`. CosineAnnealingLR over 20 epochs.

In [ ]:
model = models.resnet50(weights="IMAGENET1K_V1")

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
class_weights = class_weights.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Using device: {device}")
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
NUM_EPOCHS = 20

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    [
        {"params": model.layer4.parameters(), "lr": 1e-4},
        {"params": model.fc.parameters(),     "lr": 1e-3},
    ],
    weight_decay=1e-4,
)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_state   = None

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)
    train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)
            val_loss    += loss.item()
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss / len(train_loader))
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss / len(val_loader))
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = copy.deepcopy(model.state_dict())
        marker = "  *best*"
    else:
        marker = ""

    current_lr = scheduler.get_last_lr()[0]
    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f} | "
        f"LR(head): {current_lr:.2e}{marker}"
    )
    scheduler.step()

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = range(1, NUM_EPOCHS + 1)
axes[0].plot(epochs_x, history["train_loss"], label="train")
axes[0].plot(epochs_x, history["val_loss"],   label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(epochs_x, history["train_acc"], label="train")
axes[1].plot(epochs_x, history["val_acc"],   label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
out_path = "../models/resnet50_stylesync_occasion.pt"
torch.save(best_state, out_path)
print(f"Best occasion model (val_acc={best_val_acc:.4f}) saved to {out_path}")

In [ ]:
model.load_state_dict(torch.load(out_path, map_location=device))
model = model.to(device).eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device); labels = labels.to(device)
        preds  = model(images).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — Occasion Classifier (Validation)")
plt.show()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))